In [0]:


%pip install openpyxl

In [0]:
# sel_projects = all searches the entire project portfolio whereas sel_projects = selected will search only the manually uploaded project list
sel_projects = 'all'

# sel_hierarchy can be 'decentralization' or 'pfm' or 'asset management'
sel_hierarchy = 'pima'

#selected countries country_selected can either be 'all' or a list of countries ['Country1','Country']
country_selected = 'all'

#minimum and maximum year to be filtered
min_year = 2000
max_year = 2025

In [0]:
hierarchy_dec = {'Intergovernmental Reform and Coordination':['intergovernmental relations','intergovernmental coordination','intergovernmental reform','local self government'],
             'Functional assignment':['functional assignment','deconcentrated functions','delegated functions','subsidiarity','devolution','devolved functions','expenditure assignment'],
             'Intergovernmental fiscal architecture':['Expenditure assignment','revenue assignment','revenue sharing','equalization','finance commission','grants commission','equitable share','vertical share'],
             'Intergovernmental fiscal transfers':['intergovernmental transfers','fiscal transfers','conditional grant','specific grant','matching grant','performance grant','performance based grant', 'allocation formulae','equalization grant','finance commission','revenue sharing','local government finance','local development grant','earmarked grant'],
             'Locally raised revenue':['own source revenue','business licenses','property tax','licenses and fees','cadaster','local government tax','local revenue raising'],
             'Subnational borrowing':['local government borrowing','local government debt','municipal bonds','city bonds'],
             'Subnational PFM':['subnational PEFA','subnational budgeting'],
             'Local Human Resource Management':['local civil service','local public service'],
             'Institutional Capacity and Performance':['Local Government Performance','District Performance','municipal performance','municipal capacity','local government capacity','local capacity','capacity building grants'],
             'Local Politics':['City council','municipal council','district council','county council','local government council','local council'],
             'Participation and Accountability':['local participation','local accountability','local elections','participatory planning','participatory budgeting'],
             'local infrastructure':['local capital investment planning','local development fund','local development grant','local development plan','local investment fund','urban infrastructure','municipal infrastructure'],'Local Service Delivery':['local government services','local services','results based financing','performance based financing','district health services','district education services','municipal health services','municipal education services','municipal health services']}

hierarchy_pfm = {'PFM Reform and Diagonstic Tools':['Public Expenditure and Financial Accountabilty','PEFA','Country Policy Institutional Assessment','CPIA','PER', 'Public Expenditure Review','PEIR','Public Expenditure and Institutional Review','PFM Reform','PFMR', 'PFM Coordination'],
                 'Transparency, Participation and Accountability':['budget transparency','Open data','public participation','open budget', 'citizens guide to the budget', 'citizens budget', 'participatory budget','BOOST','Buget Portal','Budget Website','open contracting data standards','OCDS','fiscal transparency','participatory budget','participatory based budgeting','pbb','public participation'],
                 'Procurement': ['open contracting data standards','OCDS', 'E-GP','E GP', 'government procurement', 'public procurement', 'e-procurement', 'e procurement', 'procurement reform', 'green procurement'],
                 'Policy-based fiscal strategy and budgeting':['Medium Term Expenditure Framework','MTEF', 'Program-budgeting','PBB','performance budgeting','expenditure plan','Macroeconomic forecasting','Fiscal forecasting','revenue forecasting','revenue projections','Budget formulation','budget documentation','Budget preparation','resource allocation','budget allocation','budget policy','budget plan','Public expenditure','public efficiency','budget cost','gender budgeting','gender budget', 'budget tagging'],
                 'Public Investment and Asset Management':['PIM','Public Investment Appraisal','Asset Management','Infrastructure Governance','Public investment Appraisal','project evaluation','Project Appraisal Criteria','Public Investment Management Assessment','PIMA','Public Investment Plan','PIP','Asset management','asset register'],
                 'Predictability and control in budget execution':['Budget reliability','budget credibility','Budget deviation','Internal audit','Budget execution', 'Integrated Financial Management Information System','IFMIS','cash management','Cash Forecast','Cash Plan','Treasury Single Account','TSA','STA','Single Treasury Account','payment process','electronic funds transfer','EFT'],
                 'Accounting and Reporting':['Budget classification','chart of accounts','budget nomenclature','accounts nomenclature','GFS','COFOG','Financial data integrity','IPSAS','Accrual','accounting standards','Fiduciary assurance','annual financial report','annual financial statement','in-year budget report','budget execution report','bank reconciliation','accounts reconciliation'],
                 'External Scrutiny and Audit':['External audit','parliamentary oversight','legislative scrutiny','supreme audit institution','audit report','cour de comptes','finance commissions','public account commissions','public accuonts committee','parliamentary budget office','court of auditors'],
                 'Financing of Service Delivery' :  ['transfers to sub-national governments','conditional grant','performance grant','equalization grant','allocation formula','fiscal transfer','earmarked grant','specific grant','intergovernmental fiscal','subvention','capitation grant','school grant'],
                 'Climate PFM & PIM' :  ['climate tagging','climate budget tagging','sustainablity reporting','green procurement','green PIM','green public investment management','climate budget','green public procurement','climate tagging','green project appraisal']}

hierarchy_asset_management = {'Key Terms':['asset management', 'infrastructure governance', 'asset register'],
                              'Additional':['asset governance', 'public asset management', 'asset valuation']}

hierarchy_pima = {'Key Terms':['public investment management'],'Additional':['public investment management assessment','climate public investment management']}

hierarchy_drm = {'Key Terms':['domestic revenue mobilization''domestic revenue mobilisation']}

hierarchy_procurement = {'Key Terms':['procurement']}

hierarchy_tagging = {'pfm':hierarchy_pfm,'decentralization':hierarchy_dec,
                     'asset management':hierarchy_asset_management,'pima':hierarchy_pima,
                     'drm':hierarchy_drm,'procurement':hierarchy_procurement}

hierarchy = hierarchy_tagging[sel_hierarchy]
print(hierarchy)

search_term_list = []
for each_group in hierarchy:
  search_term_list.append(hierarchy[each_group])
search_term_list = [item for sublist in search_term_list for item in sublist]

print(search_term_list)

In [0]:
import pandas as pd
from pyspark.sql.functions import countDistinct
import plotly.express as px
import warnings

from pathlib import Path

warnings.simplefilter(action='ignore')

In [0]:
df_project_master = spark.read.format('csv').option("delimiter","|").option("header","true").load('dbfs:/mnt/dataanalyticslake/WB/OfficialUseOnly/CorporateData/Operations/CrossCutting/Project/PROJECT_MASTER_V3.csv')
df_project_master.createOrReplaceTempView("projMaster")

In [0]:
path_prior_actions = 'https://thedocs.worldbank.org/en/doc/4557efd6410a7d7c75bc05ad68975754-0290012025/original/DPAD-database-FY24-ext.xlsx'
df_prior_actions = pd.read_excel(path_prior_actions,sheet_name='Prior Actions database',engine='openpyxl')